# 🎓 Orientation Project — Academic ML Pipeline (One Notebook)

This notebook implements the complete, professor-approved Machine Learning workflow for student specialization prediction.

- One file for ALL data handling and ML steps
- No training before data cleaning
- Every step is visible, justified, and visualized
- Outputs are saved to `/reports` for the admin dashboard




## 1. Introduction & Context

- Objective: Predict the best specialization for a student from test and skill data
- Importance: Data preparation is critical; we strictly avoid training before cleaning
- Lifecycle: Data → Quality control → EDA → Cleaning → Preprocessing → Split → Modeling → Evaluation → Export


## 2. Problem Definition
- Type: Supervised multi-class classification (18 classes)
- Target: `specialization_label`
- Inputs: `q1_score..q25_score`, `practical_test_score`, `logical_reasoning_score`, `problem_solving_score`, `time_spent_minutes`, `previous_level`, `current_level`, `improvement_rate`, `filiere`
- Hypotheses:
  - Cleaning and normalization improve performance
  - Stratified split is required due to imbalance
  - Ensemble models likely outperform linear baselines


In [ ]:
# 3. Setup & Imports
import os, json, sqlite3, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import joblib

sns.set_theme(style='whitegrid')

# Directories
os.makedirs('reports', exist_ok=True)
os.makedirs('reports/figures', exist_ok=True)
os.makedirs('reports/model', exist_ok=True)
os.makedirs('reports/data', exist_ok=True)

DB_PATH = 'specialization_prediction.db'
RAW_TABLE = 'raw_student_tests'
CLEAN_TABLE = 'clean_student_tests'

print('[OK] Environment prepared')


In [ ]:
# 4. Data Loading / Generation

# Try load from SQLite; else generate CSV then persist
conn = sqlite3.connect(DB_PATH)
existing_tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

def load_from_sqlite(table):
    try:
        return pd.read_sql(f'SELECT * FROM {table}', conn)
    except Exception:
        return None

# Attempt to load RAW data
df = load_from_sqlite(RAW_TABLE)

if df is None or len(df) == 0:
    # Generate with existing generator (ensures 50k rows and injected issues)
    from dataset_generator import generate_dataset
    print('[INFO] No RAW table found; generating synthetic dataset (~50k rows)')
    df = generate_dataset(n_samples=50000)
    # Persist RAW to CSV + SQLite
    df.to_csv('reports/data/raw_student_tests.csv', index=False)
    df.to_sql(RAW_TABLE, conn, if_exists='replace', index=False)
    print('[OK] RAW dataset saved to SQLite and CSV')
else:
    print(f'[OK] Loaded RAW from SQLite: {df.shape}')
    df.to_csv('reports/data/raw_student_tests.csv', index=False)

# Basic peek
display(df.head())
print('Shape:', df.shape)


In [ ]:
# 5. Dataset Description & Quality Control

n_rows, n_cols = df.shape
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_total = int(missing_counts.sum())
duplicates_count = int(df.duplicated().sum())

print({'rows': n_rows, 'cols': n_cols, 'missing_total': missing_total, 'duplicates': duplicates_count})

# KPI-like summary dataframe
kpi = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Total Missing', 'Duplicates'],
    'Value': [n_rows, n_cols, missing_total, duplicates_count]
})
display(kpi)

# Save KPI JSON for dashboard
with open('reports/kpi.json', 'w', encoding='utf-8') as f:
    json.dump({'rows': n_rows, 'cols': n_cols, 'missing_total': missing_total, 'duplicates': duplicates_count}, f, indent=2)

# Missing values bar chart
if (missing_counts > 0).any():
    plt.figure(figsize=(14, 8))
    top_missing = missing_counts[missing_counts > 0].head(25)
    top_missing.plot(kind='barh', color='#667eea', edgecolor='black')
    plt.gca().invert_yaxis()
    plt.title('Top Missing Values per Column')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.savefig('reports/figures/missing_values.png', dpi=200)
    plt.show()

# Class distribution
if 'specialization_label' in df.columns:
    plt.figure(figsize=(16, 8))
    df['specialization_label'].value_counts().plot(kind='bar', color='#764ba2', edgecolor='black')
    plt.title('Class Distribution — specialization_label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('reports/figures/class_distribution.png', dpi=200)
    plt.show()


In [ ]:
# 6. EDA — BEFORE Cleaning

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Descriptive statistics
stats = df[num_cols].describe().T
stats.to_csv('reports/descriptive_stats.csv')
display(stats.head(10))

# Histograms (subset for speed)
plot_cols = [c for c in num_cols if c not in ['user_id']][:12]
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, c in enumerate(plot_cols):
    axes[i].hist(df[c].dropna(), bins=50, color='#4facfe', edgecolor='black', alpha=0.8)
    axes[i].set_title(c, fontsize=10)
plt.tight_layout()
plt.savefig('reports/figures/histograms.png', dpi=200)
plt.show()

# Boxplots for outliers (scores only)
score_cols = [c for c in num_cols if 'score' in c.lower()][:12]
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, c in enumerate(score_cols):
    axes[i].boxplot(df[c].dropna(), vert=True)
    axes[i].set_title(c, fontsize=10)
plt.tight_layout()
plt.savefig('reports/figures/boxplots.png', dpi=200)
plt.show()

# Correlation heatmap (subset)
corr_cols = ['practical_test_score', 'logical_reasoning_score', 'problem_solving_score', 'time_spent_minutes'] + [f'q{i}_score' for i in range(1, 9)]
corr_cols = [c for c in corr_cols if c in df.columns]
plt.figure(figsize=(14, 10))
sns.heatmap(df[corr_cols].corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (subset)')
plt.tight_layout()
plt.savefig('reports/figures/correlation_heatmap.png', dpi=200)
plt.show()


In [ ]:
# 7. Data Preprocessing (Cleaning → Encoding → Scaling)

# 7.A Remove duplicates
before_rows = len(df)
df_clean = df.drop_duplicates().copy()

dups_removed = before_rows - len(df_clean)
print({'duplicates_removed': int(dups_removed)})

# 7.B Fix invalid values and ranges
question_cols = [f'q{i}_score' for i in range(1, 26) if f'q{i}_score' in df_clean.columns]
for c in question_cols:
    df_clean[c] = df_clean[c].clip(0, 3)
for c in ['practical_test_score','logical_reasoning_score','problem_solving_score']:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].clip(0, 100)
if 'time_spent_minutes' in df_clean.columns:
    df_clean['time_spent_minutes'] = df_clean['time_spent_minutes'].clip(lower=0)

# 7.C Missing values: COMPLETE IMPUTATION TO REACH 0 MISSING VALUES
print('[7.C] Handling ALL missing values...')
missing_before = df_clean.isnull().sum().sum()
print(f'  Missing values before: {missing_before:,}')

# Strategy 1: Mean imputation for question scores
for c in question_cols:
    if df_clean[c].isna().any():
        mean_val = df_clean[c].mean()
        if pd.isna(mean_val):  # If all values are NaN, use 0
            mean_val = 0
        df_clean[c] = df_clean[c].fillna(mean_val)

# Strategy 2: KNN imputation for other numeric columns
num_impute = [c for c in ['practical_test_score','logical_reasoning_score','problem_solving_score','time_spent_minutes','previous_level','current_level','improvement_rate'] if c in df_clean.columns]
if df_clean[num_impute].isna().sum().sum() > 0:
    knn = KNNImputer(n_neighbors=5)
    df_clean[num_impute] = knn.fit_transform(df_clean[num_impute])

# Strategy 3: Handle ANY remaining numeric missing values with mean/median
numeric_cols_all = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols_all:
    if df_clean[col].isna().any():
        # Use median if mean is NaN, otherwise use mean
        fill_val = df_clean[col].median() if not pd.isna(df_clean[col].median()) else df_clean[col].mean()
        if pd.isna(fill_val):  # If still NaN, use 0
            fill_val = 0
        missing_count = df_clean[col].isna().sum()
        df_clean[col] = df_clean[col].fillna(fill_val)
        if missing_count > 0:
            print(f'  - {col}: Filled {missing_count} missing values with {fill_val:.2f}')

# Strategy 4: Handle categorical missing values with mode (most frequent)
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    if df_clean[col].isna().any():
        mode_val = df_clean[col].mode()
        if len(mode_val) > 0:
            fill_val = mode_val[0]
        else:
            # If no mode (all NaN), use first non-null value or default
            non_null = df_clean[col].dropna()
            fill_val = non_null.iloc[0] if len(non_null) > 0 else 'Unknown'
        missing_count = df_clean[col].isna().sum()
        df_clean[col] = df_clean[col].fillna(fill_val)
        if missing_count > 0:
            print(f'  - {col}: Filled {missing_count} missing values with mode "{fill_val}"')

# Final verification: ensure 0 missing values - AGGRESSIVE CLEANUP
missing_after = df_clean.isnull().sum().sum()
print(f'\n  After strategies 1-4, remaining missing: {missing_after:,}')

if missing_after > 0:
    print(f'  WARNING: Still {missing_after} missing values. Applying AGGRESSIVE final cleanup...')
    # Last resort: fill ANY remaining NaN with appropriate defaults
    for col in df_clean.columns:
        if df_clean[col].isna().any():
            missing_count = df_clean[col].isna().sum()
            # Check dtype more carefully
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                # For numeric: use 0 if all are NaN, else use median/mean
                if df_clean[col].notna().sum() == 0:
                    fill_val = 0
                else:
                    fill_val = df_clean[col].median() if not pd.isna(df_clean[col].median()) else df_clean[col].mean()
                    if pd.isna(fill_val):
                        fill_val = 0
                df_clean[col] = df_clean[col].fillna(fill_val)
                print(f'    - {col} (numeric): Filled {missing_count} with {fill_val:.2f}')
            else:
                # For categorical: use mode or first non-null or 'Unknown'
                non_null = df_clean[col].dropna()
                if len(non_null) > 0:
                    fill_val = df_clean[col].mode()[0] if len(df_clean[col].mode()) > 0 else non_null.iloc[0]
                else:
                    fill_val = 'Unknown'
                df_clean[col] = df_clean[col].fillna(fill_val)
                print(f'    - {col} (categorical): Filled {missing_count} with "{fill_val}"')
    
    # Double check - if still any NaN, force fill
    missing_final = df_clean.isnull().sum().sum()
    if missing_final > 0:
        print(f'  CRITICAL: Still {missing_final} missing after aggressive cleanup. Forcing fill...')
        # Force fill everything
        df_clean = df_clean.fillna(0)  # Fill numeric with 0
        df_clean = df_clean.fillna('Unknown')  # Fill remaining (categorical) with 'Unknown'
        missing_final = df_clean.isnull().sum().sum()
        print(f'  After force fill, missing: {missing_final}')
    
    if missing_final == 0:
        print(f'  ✅ SUCCESS: All missing values eliminated! Final count: {missing_final}')
    else:
        print(f'  ❌ ERROR: Still {missing_final} missing values after all attempts!')
else:
    print(f'  ✅ SUCCESS: All missing values imputed! Final count: {missing_after}')
    missing_final = missing_after

# 7.D Outliers (IQR) for time
if 'time_spent_minutes' in df_clean.columns:
    Q1, Q3 = df_clean['time_spent_minutes'].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    df_clean['time_spent_minutes'] = df_clean['time_spent_minutes'].clip(lower, upper)

# 7.E Encode categoricals
enc = LabelEncoder()
if 'filiere' in df_clean.columns:
    df_clean['filiere_encoded'] = enc.fit_transform(df_clean['filiere'].astype(str))

# 7.F Final verification and summary - VERIFY 0 MISSING VALUES
missing_final = df_clean.isnull().sum().sum()
rows_after = len(df_clean)
rows_before = before_rows

# Show per-column missing values if any remain
if missing_final > 0:
    print('\n  ⚠️  WARNING: Missing values per column:')
    missing_per_col = df_clean.isnull().sum()
    missing_per_col = missing_per_col[missing_per_col > 0]
    for col, count in missing_per_col.items():
        print(f'    - {col}: {count} missing')

print('\n[7.F] Cleaning Summary:')
print(f'  Rows: {rows_before:,} → {rows_after:,} (removed {rows_before - rows_after:,} duplicates)')
print(f'  Missing values: {missing_before:,} → {missing_final:,} (removed {missing_before - missing_final:,})')
if missing_final == 0:
    print(f'  ✅ TARGET ACHIEVED: 0 missing values!')
else:
    print(f'  ❌ TARGET NOT MET: {missing_final} missing values remaining')
print(f'  Columns: {len(df_clean.columns)}')

# Save cleaning summary for dashboard
cleaning_summary = pd.DataFrame({
    'Metric': ['Rows', 'Missing Values', 'Duplicates'],
    'Before': [rows_before, missing_before, int(df.duplicated().sum())],
    'After': [rows_after, missing_final, 0],
    'Difference': [rows_after - rows_before, missing_final - missing_before, -int(df.duplicated().sum())]
})
cleaning_summary.to_csv('reports/cleaning_summary.csv', index=False)
print('[OK] Cleaning summary saved')

# Save CLEANED to CSV + SQLite
clean_path = 'reports/data/clean_student_tests.csv'
df_clean.to_csv(clean_path, index=False)
df_clean.to_sql(CLEAN_TABLE, conn, if_exists='replace', index=False)
print('[OK] CLEANED saved to SQLite and CSV')

# Final absolute verification
missing_absolute_final = df_clean.isnull().sum().sum()
print(f'\n[VERIFIED] Final missing values count: {missing_absolute_final} (target: 0)')
if missing_absolute_final == 0:
    print('✅ SUCCESS: Dataset has 0 missing values! Ready for ML training.')
else:
    print(f'❌ WARNING: Dataset still has {missing_absolute_final} missing values!')
    print('   This should not happen. Please check the imputation code.')


In [ ]:
# 8. Feature Engineering

df_feat = df_clean.copy()
# Aggregations
if set(question_cols).issubset(df_feat.columns):
    df_feat['avg_q_score'] = df_feat[question_cols].mean(axis=1)
    df_feat['sum_q_score'] = df_feat[question_cols].sum(axis=1)
# Composite indicators
if {'practical_test_score','logical_reasoning_score','problem_solving_score'}.issubset(df_feat.columns):
    df_feat['overall_skill'] = df_feat[['practical_test_score','logical_reasoning_score','problem_solving_score']].mean(axis=1)

print('[OK] Engineered features:', [c for c in ['avg_q_score','sum_q_score','overall_skill'] if c in df_feat.columns])


In [ ]:
# 9. Train / Test Split (Stratified)

feature_cols = [c for c in (question_cols + ['practical_test_score','logical_reasoning_score','problem_solving_score','time_spent_minutes','previous_level','current_level','improvement_rate','filiere_encoded','avg_q_score','sum_q_score','overall_skill']) if c in df_feat.columns]
X = df_feat[feature_cols].copy()
y = df_feat['specialization_label'].astype(str)

y_enc = LabelEncoder().fit_transform(y)
class_names = sorted(list(pd.Series(y).unique()))

X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('Train:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
# 10. Model Selection & Training (CV)

models = {
    'LogReg': LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=150, max_depth=25, n_jobs=-1, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(128,64), max_iter=50, random_state=42)
}

results = {}

for name, model in models.items():
    needs_scale = name in ['LogReg','SVM','MLP']
    X_tr = X_train_s if needs_scale else X_train
    X_te = X_test_s if needs_scale else X_test
    cv = cross_val_score(model, X_tr, y_train, cv=3, scoring='accuracy', n_jobs=-1)
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    results[name] = {
        'acc': accuracy_score(y_test, y_pred),
        'prec': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'rec': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'cv_mean': float(cv.mean()),
        'cv_std': float(cv.std()),
        'y_pred': y_pred,
        'needs_scale': needs_scale,
        'model': model
    }

comp = pd.DataFrame([{ 'Model': k, **{ 'Accuracy': v['acc'], 'Precision': v['prec'], 'Recall': v['rec'], 'F1': v['f1'], 'CV_mean': v['cv_mean'], 'CV_std': v['cv_std'] } } for k,v in results.items()]).sort_values('Accuracy', ascending=False)
display(comp)
comp.to_csv('reports/model_comparison.csv', index=False)

plt.figure(figsize=(10,6))
plt.bar(comp['Model'], comp['Accuracy'], color=['#667eea','#764ba2','#4facfe','#00c6ff'])
plt.title('Model Accuracy')
plt.ylim(0,1)
plt.tight_layout()
plt.savefig('reports/figures/model_accuracy.png', dpi=200)
plt.show()


In [ ]:
# 11. Evaluation & Confusion Matrices

best_name = comp.iloc[0]['Model']
best = results[best_name]

from math import ceil
n = len(results)
cols = min(2, n)
rows = ceil(n/cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, 5*rows))
axes = np.array(axes).reshape(-1) if n>1 else [axes]

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, ax=axes[i], cmap='Blues', cbar=False)
    axes[i].set_title(f'{name} — Acc: {res["acc"]:.3f}')
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')

plt.tight_layout()
plt.savefig('reports/figures/confusion_matrices.png', dpi=200)
plt.show()

metrics = { m: {k: float(v[m]) if isinstance(v[m], (int,float,np.floating)) else v[m] for k,v in results.items()} for m in ['acc','prec','rec','f1','cv_mean','cv_std'] }
metrics['best_model'] = best_name
with open('reports/metrics.json','w',encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('[OK] Saved metrics.json and figures')


In [ ]:
# 12. Model Improvement (Light Tuning for Best)

tuned = None
if best_name == 'RandomForest':
    grid = {'n_estimators':[150,250], 'max_depth':[20,30]}
    gs = GridSearchCV(RandomForestClassifier(n_jobs=-1, random_state=42), grid, cv=3, scoring='accuracy', n_jobs=-1)
    gs.fit(X_train, y_train)
    tuned = gs.best_estimator_
elif best_name == 'SVM':
    grid = {'C':[0.5,1.0], 'gamma':['scale','auto']}
    gs = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=42), grid, cv=3, scoring='accuracy', n_jobs=-1)
    gs.fit(X_train_s, y_train)
    tuned = gs.best_estimator_
elif best_name == 'LogReg':
    grid = {'C':[0.5,1.0,2.0]}
    gs = GridSearchCV(LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42), grid, cv=3, scoring='accuracy', n_jobs=-1)
    gs.fit(X_train_s, y_train)
    tuned = gs.best_estimator_

if tuned is not None:
    Xt = X_test_s if best_name in ['LogReg','SVM','MLP'] else X_test
    tuned.fit(X_train_s if best_name in ['LogReg','SVM','MLP'] else X_train, y_train)
    y_pred_t = tuned.predict(Xt)
    tuned_acc = accuracy_score(y_test, y_pred_t)
    print({'original_acc': float(best['acc']), 'tuned_acc': float(tuned_acc)})
else:
    print('[INFO] Light tuning not applied for this best model')


In [ ]:
# 13. Final Export (Model + Preprocessing + Figures)

export = {
    'best_model': best_name,
    'feature_cols': feature_cols,
    'class_names': class_names
}

joblib.dump({'model': results[best_name]['model'], 'scaler': scaler if results[best_name]['needs_scale'] else None, 'feature_cols': feature_cols, 'class_names': class_names}, 'reports/model/best_model.pkl')

with open('reports/export_summary.json','w',encoding='utf-8') as f:
    json.dump(export, f, indent=2)

print('[OK] Exported model and summary to /reports/model and /reports')


In [ ]:
# 14. Clean vs Raw Performance Comparison (Why Cleaning Matters)

# Minimal baseline on RAW (no cleaning, only simple fill & encoding to avoid crashes)
df_raw_baseline = df.copy()
# Safe fill
for c in [f'q{i}_score' for i in range(1,26) if f'q{i}_score' in df_raw_baseline.columns]:
    df_raw_baseline[c] = df_raw_baseline[c].fillna(df_raw_baseline[c].mean())
for c in ['practical_test_score','logical_reasoning_score','problem_solving_score','time_spent_minutes','previous_level','current_level','improvement_rate']:
    if c in df_raw_baseline.columns:
        df_raw_baseline[c] = df_raw_baseline[c].fillna(df_raw_baseline[c].median())
if 'filiere' in df_raw_baseline.columns:
    df_raw_baseline['filiere_encoded'] = LabelEncoder().fit_transform(df_raw_baseline['filiere'].astype(str))

feat_raw = [c for c in feature_cols if c in df_raw_baseline.columns]
Xr = df_raw_baseline[feat_raw].copy(); yr = df_raw_baseline['specialization_label'].astype(str)
yr_enc = LabelEncoder().fit_transform(yr)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr_enc, stratify=yr_enc, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
rf.fit(Xr_tr, yr_tr)
raw_acc = accuracy_score(yr_te, rf.predict(Xr_te))

clean_rf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
clean_rf.fit(X_train, y_train)
clean_acc = accuracy_score(y_test, clean_rf.predict(X_test))

print({'raw_acc': float(raw_acc), 'clean_acc': float(clean_acc)})

plt.figure(figsize=(6,4))
plt.bar(['RAW','CLEAN'], [raw_acc, clean_acc], color=['#e53e3e','#38a169'])
plt.ylim(0,1); plt.ylabel('Accuracy'); plt.title('Performance: RAW vs CLEANED')
plt.tight_layout(); plt.savefig('reports/figures/raw_vs_clean.png', dpi=200); plt.show()

with open('reports/clean_vs_raw.json','w',encoding='utf-8') as f:
    json.dump({'raw_acc': float(raw_acc), 'clean_acc': float(clean_acc)}, f, indent=2)


## 15. Conclusion & Perspectives
- Best model and key metrics are exported to `/reports`
- Cleaning significantly improves performance (see `raw_vs_clean.png`)
- Next steps: richer feature engineering, imbalance mitigation (class weights), and calibration

All outputs are ready for the admin dashboard:
